In [ ]:
# =========================
# 0) Install packages
# =========================
!apt-get install -y -qq fonts-nanum
!pip install -q -U tensorflow scikit-learn pandas matplotlib

# =========================
# 1) Imports & settings
# =========================
import os, unicodedata
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, BatchNormalization, Dropout, Input
from tensorflow.keras.applications import EfficientNetB0
from google.colab import drive

# Use an English font for all figures
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# Google Drive
drive.mount('/content/drive')

# =========================
# 2) Paths
# =========================
base_dir  = '/content/drive/MyDrive/Colab Notebooks/testset'
image_dir = os.path.join(base_dir, 'TS_Color_위해물품')

# =========================
# 3) Collect image paths / labels
# =========================
image_paths, labels = [], []
valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

class_labels = sorted([d for d in os.listdir(image_dir) if os.path.isdir(os.path.join(image_dir, d))])
class_labels = [unicodedata.normalize('NFC', lb) for lb in class_labels]
num_classes  = len(class_labels)
print(f"Classes ({num_classes}): {class_labels}")

for cl in class_labels:
    cdir = os.path.join(image_dir, cl)
    for name in os.listdir(cdir):
        if name.lower().endswith(valid_ext):
            image_paths.append(os.path.join(cdir, name))
            labels.append(unicodedata.normalize('NFC', cl))

data_df = pd.DataFrame({'filename': image_paths, 'label': labels})

# Label encoding & class weights
label_encoder = LabelEncoder()
data_df['label_encoded'] = label_encoder.fit_transform(data_df['label'])

train_df, temp_df = train_test_split(data_df, test_size=0.3, random_state=42, stratify=data_df['label'])
val_df,   test_df = train_test_split(temp_df, test_size=0.33, random_state=42, stratify=temp_df['label'])

for df in (train_df, val_df, test_df):
    df.reset_index(drop=True, inplace=True)

classes = np.unique(train_df['label_encoded'])
cw = compute_class_weight(class_weight='balanced', classes=classes, y=train_df['label_encoded'])
cw = np.clip(cw, 0, 5.0)
class_weight_dict = dict(zip(classes, cw))
print("Class weights:", class_weight_dict)

# =========================
# 4) Generators
# =========================
batch_size = 32
datagen_train = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45, width_shift_range=0.3, height_shift_range=0.3,
    shear_range=0.3, zoom_range=0.3, horizontal_flip=True, fill_mode='nearest'
)
datagen_val_test = ImageDataGenerator(rescale=1./255)

class CustomDataGenerator(Sequence):
    def __init__(self, df, batch_size, is_training, datagen):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.is_training = is_training
        self.datagen = datagen
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()
    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))
    def __getitem__(self, idx):
        idxs = self.indices[idx*self.batch_size : (idx+1)*self.batch_size]
        part = self.df.iloc[idxs]
        imgs = []
        for fn in part['filename']:
            im = tf.keras.preprocessing.image.load_img(fn, target_size=(224, 224))
            im = tf.keras.preprocessing.image.img_to_array(im)
            imgs.append(im)
        imgs = np.array(imgs)
        if self.is_training:
            it = self.datagen.flow(imgs, batch_size=len(imgs), shuffle=False)
            imgs = next(it)
        else:
            imgs = imgs / 255.0
        y = label_encoder.transform(part['label'])
        y = tf.keras.utils.to_categorical(y, num_classes=num_classes)
        return imgs, y
    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.indices)

train_gen = CustomDataGenerator(train_df, batch_size, True,  datagen_train)
val_gen   = CustomDataGenerator(val_df,   batch_size, False, datagen_val_test)
test_gen  = CustomDataGenerator(test_df,  batch_size, False, datagen_val_test)

steps_per_epoch  = len(train_gen)
validation_steps = len(val_gen)
test_steps       = len(test_gen)

# =========================
# 5) English class-name mapping for reports/plots
# =========================
k2e = {
    'TS_Color_위해물품_Bullet_총알'                        : 'Bullet',
    'TS_Color_위해물품_Card knife_카드칼(접은 형태)(167)'   : 'Card knife',
    'TS_Color_위해물품_Electroshock weapon_전자충격기'      : 'Stun gun',
    'TS_Color_위해물품_Grenade_수류탄'                     : 'Grenade',
    'TS_Color_위해물품_Knife blade_칼날(030)'              : 'Blade',
    'TS_Color_위해물품_Knife-D_맥가이버 칼(접은 형태)(029)' : 'Swiss army knife',
    'TS_Color_위해물품_Knife-E_펜칼'                       : 'Pen knife',
    'TS_Color_위해물품_Pistol_모형 권총(금속 재질)'          : 'Metal pistol',
    'TS_Color_위해물품_Pistol_모형 권총(플라스틱 재질)'      : 'Plastic pistol',
    'TS_Color_위해물품_Rifle_모형 소총(플라스틱 재질)'       : 'Plastic rifle',
    'TS_Color_위해물품_Scissors-A_문구용 가위(접은 형태)(037)': 'Scissors',
}
k2e = {unicodedata.normalize('NFC', k): v for k, v in k2e.items()}

class_indices = {v: k for k, v in enumerate(label_encoder.classes_)}
inv_class_indices = {v: k for k, v in class_indices.items()}
idx2name_en = [k2e.get(inv_class_indices[i], inv_class_indices[i]) for i in range(num_classes)]

# =========================
# 6) Model (fixed hyperparameters)
# =========================
def build_fixed_model():
    image_input = Input(shape=(224, 224, 3), name='image_input')

    base = EfficientNetB0(input_tensor=image_input, include_top=False, weights='imagenet')
    base.trainable = True
    # freeze early layers
    for l in base.layers[:20]:
        l.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)

    # ---- Fixed hyperparameters (as requested) ----
    # Best hyperparameters:
    # - units_dense1: 256
    # - dropout_rate1: 0.3
    # - units_dense2: 384
    # - dropout_rate2: 0.3
    # - learning_rate: 0.00021395496476078402
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(384, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=image_input, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.00021395496476078402),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_fixed_model()
model.summary()

# =========================
# 7) Train
# =========================
ckpt_cb = ModelCheckpoint("/content/drive/MyDrive/Colab Notebooks/checkpoints/model_efficientnetb0_fixed_en.h5",
                          save_best_only=True)
rlr_cb  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
es_cb   = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=50,
    callbacks=[ckpt_cb, rlr_cb, es_cb],
    class_weight=class_weight_dict
)

# =========================
# 8) Learning curves (EN)
# =========================
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'],     label='Train accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.title('Training & Validation Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'],     label='Train loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.title('Training & Validation Loss'); plt.legend()
plt.show()

# =========================
# 9) Evaluation / Predictions / Report / Confusion Matrix / ROC (EN)
# =========================
test_loss, test_acc = model.evaluate(test_gen, steps=test_steps)
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

y_pred_probs   = model.predict(test_gen, steps=test_steps)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

y_true_classes = []
for i in range(test_steps):
    _, by = test_gen[i]
    y_true_classes.extend(np.argmax(by, axis=1))
y_true_classes = np.array(y_true_classes)

# Classification report in English
print(classification_report(y_true_classes, y_pred_classes, target_names=idx2name_en, digits=4))

# Confusion matrix in English
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(12,12))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=idx2name_en)\
    .plot(cmap=plt.cm.Blues, xticks_rotation=90, values_format='d')
plt.title('Confusion Matrix'); plt.xlabel('Predicted label'); plt.ylabel('True label')
plt.show()

# ROC curves in English
from sklearn.preprocessing import label_binarize
y_true_bin = label_binarize(y_true_classes, classes=range(num_classes))
y_pred_probs = y_pred_probs[:len(y_true_bin)]

fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(10,8))
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label=f'{idx2name_en[i]} (AUC = {roc_auc[i]:.2f})')
plt.plot([0,1],[0,1],'k--')
plt.title('Multi-class ROC'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.legend(loc='best'); plt.show()

# =========================
# 10) Correct / Incorrect sample visualization (EN)
# =========================
x_test, y_test_cls = [], []
for i in range(test_steps):
    bx, by = test_gen[i]
    x_test.extend(bx); y_test_cls.extend(np.argmax(by, axis=1))
x_test = np.array(x_test); y_test_cls = np.array(y_test_cls)

correct_idx   = np.nonzero(y_pred_classes == y_true_classes)[0]
incorrect_idx = np.nonzero(y_pred_classes != y_true_classes)[0]
pred_names = [idx2name_en[i] for i in y_pred_classes]
true_names = [idx2name_en[i] for i in y_true_classes]

plt.figure(figsize=(10,10))
for i, idx in enumerate(correct_idx[:9]):
    plt.subplot(3,3,i+1); plt.imshow((x_test[idx]*255).astype('uint8'))
    plt.title(f"GT: {true_names[idx]}\nPred: {pred_names[idx]}"); plt.axis('off')
plt.tight_layout(); plt.show()

plt.figure(figsize=(10,10))
for i, idx in enumerate(incorrect_idx[:9]):
    plt.subplot(3,3,i+1); plt.imshow((x_test[idx]*255).astype('uint8'))
    plt.title(f"GT: {true_names[idx]}\nPred: {pred_names[idx]}"); plt.axis('off')
plt.tight_layout(); plt.show()


Selecting previously unselected package fonts-nanum.
(Reading database ... 122214 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 136.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 134.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,485,550 (17.11 MB)

 Trainable params: 4,436,863 (16.93 MB)

 Non-trainable params: 48,687 (190.19 KB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


KeyboardInterrupt: 